# 05_tune_nsga2: Step 5 - tune NSGA-II hyperparameters with PSO, GA and HHO (Lyu et al. used Optuna).

```
Search space (paper Table A.8 widened, + setpoint mutation sigma):
  pop_size 10-100, mutation prob 0.05-1, wall mutation sd eta_m 0.2-6 m,
  crossover prob 0.1-1, setpoint mutation sd sigma_t 0.05-1.5 degC.
Fitness = -mean normalised hypervolume over 3 tuning seeds (101-103) at a FIXED
budget of function evaluations, so a bigger population cannot win by simply
spending more evaluations. Tuning seeds are disjoint from the final seeds 1-30.
Script asli: `scripts/05_tune_nsga2.py` [--budget 50] [--mh-seeds 2] [--nsga-evals 3000] — parameter kini diatur di sel `args` di bawah.
```

Dikonversi dari `scripts/05_tune_nsga2.py`. Jalankan dari folder `notebooks/`.

In [1]:
from types import SimpleNamespace
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

ROOT = Path.cwd().resolve().parent   # notebook runs from AI-OBEP-MLPaper/notebooks
sys.path.insert(0, str(ROOT / "src"))
from obep_mlpaper.layout import case_study_1, case_study_2
from obep_mlpaper.metaheuristics import ALGORITHMS, Param, SearchSpace
from obep_mlpaper.nsga import run_nsga2
from obep_mlpaper.surrogate import Surrogate

# Parameters (were command-line arguments in the script; edit here)
args = SimpleNamespace(budget=50, mh_seeds=2, nsga_evals=3000, pop=10)

SPACE = SearchSpace([
    Param("pop_size", 10, 100, "int"),
    Param("p_m", 0.05, 1.0),
    Param("eta_m", 0.2, 6.0, "log"),
    Param("p_c", 0.1, 1.0),
    Param("sigma_t", 0.05, 1.5, "log"),
])
TUNE_SEEDS = (101, 102, 103)
sur = Surrogate.load(ROOT / "outputs" / "models" / "surrogate.joblib").single_thread()
out = ROOT / "outputs" / "nsga2"
out.mkdir(parents=True, exist_ok=True)

In [2]:
result = {}
runs, traces = [], {}
for sc in (case_study_1(), case_study_2()):
    def fitness(hp, sc=sc):
        hvs = Parallel(n_jobs=2)(delayed(run_nsga2)(sc, sur, hp, s, args.nsga_evals) for s in TUNE_SEEDS)
        return -float(np.mean([r["hv"] for r in hvs]))

    for algo, fn in ALGORITHMS.items():
        for seed in range(args.mh_seeds):
            t0 = time.time()
            kw = {"PSO": "n_particles", "GA": "pop_size", "HHO": "n_hawks"}[algo]
            r = fn(fitness, SPACE, budget=args.budget, seed=seed, **{kw: args.pop})
            runs.append(dict(case=sc.name, algo=algo, seed=seed, best_hv=-r["best_f"],
                             minutes=(time.time() - t0) / 60, params=json.dumps(r["best_params"])))
            traces[f"{sc.name}_{algo}_s{seed}"] = -r["trace"]
            print(f"{sc.name} {algo} seed={seed}: HV={-r['best_f']:.4f} {r['best_params']} "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)

runs = pd.DataFrame(runs)
runs.to_csv(out / "nsga2_tuning_runs.csv", index=False)
L = max(len(t) for t in traces.values())
pd.DataFrame({k: np.pad(t, (0, L - len(t)), mode="edge") for k, t in traces.items()}).to_csv(
    out / "nsga2_tuning_convergence.csv", index_label="evaluation")
for case, grp in runs.groupby("case"):
    result[case] = {a: json.loads(grp[grp.algo == a].sort_values("best_hv").iloc[-1].params) for a in ALGORITHMS}
(out / "nsga2_tuned_hyperparameters.json").write_text(json.dumps(result, indent=2))
print(runs.groupby(["case", "algo"]).best_hv.agg(["mean", "std", "max"]).round(4))

case1_simple PSO seed=0: HV=1.0076 {'pop_size': 29, 'p_m': 0.13264738596981773, 'eta_m': 1.6974466534249653, 'p_c': 0.6156676572903119, 'sigma_t': 0.4392805491134295} (1.1 min)


case1_simple PSO seed=1: HV=1.0010 {'pop_size': 30, 'p_m': 0.1990722583756357, 'eta_m': 4.197545893317746, 'p_c': 0.9269093115900524, 'sigma_t': 0.6263124217167159} (1.0 min)


case1_simple GA seed=0: HV=1.0023 {'pop_size': 25, 'p_m': 0.16386801850379856, 'eta_m': 3.2802093863913666, 'p_c': 0.6861418276032198, 'sigma_t': 0.40825479343467114} (1.2 min)


case1_simple GA seed=1: HV=0.9981 {'pop_size': 56, 'p_m': 0.13171444433865837, 'eta_m': 1.5187659334114691, 'p_c': 0.7932798068010063, 'sigma_t': 0.43371434161768135} (0.8 min)


case1_simple HHO seed=0: HV=1.0016 {'pop_size': 42, 'p_m': 0.2561676216921329, 'eta_m': 0.22095626952762257, 'p_c': 0.13785318920049236, 'sigma_t': 0.43909268843330224} (1.3 min)


...

case2_watson GA seed=0: HV=1.1051 {'pop_size': 13, 'p_m': 0.1716243699477339, 'eta_m': 1.950368662782719, 'p_c': 0.5358031098554626, 'sigma_t': 0.40337511231602086} (1.4 min)


case2_watson GA seed=1: HV=1.0868 {'pop_size': 36, 'p_m': 0.2586744701924428, 'eta_m': 0.8325607165697342, 'p_c': 0.8266550437648348, 'sigma_t': 0.4033027629317029} (0.9 min)


case2_watson HHO seed=0: HV=1.1038 {'pop_size': 19, 'p_m': 0.05, 'eta_m': 1.3139634511948692, 'p_c': 0.6478849101626074, 'sigma_t': 0.6446493034675068} (1.9 min)


case2_watson HHO seed=1: HV=1.0884 {'pop_size': 44, 'p_m': 0.0723663211629808, 'eta_m': 2.334712278894981, 'p_c': 1.0, 'sigma_t': 1.3149470025978944} (1.2 min)


                     mean     std     max
case         algo                        
case1_simple GA    1.0002  0.0029  1.0023
             HHO   1.0015  0.0001  1.0016
             PSO   1.0043  0.0046  1.0076
case2_watson GA    1.0960  0.0129  1.1051
             HHO   1.0961  0.0109  1.1038
             PSO   1.0996  0.0070  1.1046
